In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
encounter_df = spark.table("db_healthcare_kl.gold.encounter_summaries")

cohort_df = spark.table("db_healthcare_kl.gold.patient_cohorts")

readmission_df = spark.table("db_healthcare_kl.gold.readmission_within_30days")

person_df = spark.table("db_healthcare_kl.omop.person_data")

visit_df = spark.table("db_healthcare_kl.omop.visit_occurrence")

condition_df = spark.table("db_healthcare_kl.omop.condition_occurrence")

drug_df = spark.table("db_healthcare_kl.omop.drug_exposure")

In [0]:
person_df.printSchema()

visit_df.printSchema()

condition_df.printSchema()

drug_df.printSchema()

In [0]:
person_features = (
    person_df
    .withColumn("date_of_birth", to_date("date_of_birth"))
    .withColumn(
        "age",
        year(current_date()) - year(col("date_of_birth"))
    )
    .select(
        "person_id",
        "age",
        "gender_concept_id"
    )
)

display(person_features)

In [0]:
diagnosis_features = (
    condition_df
    .groupBy("person_id")
    .agg(
        count("*").alias("diagnosis_count")
    )
)

display(diagnosis_features)

In [0]:
medication_features = (
    drug_df
    .groupBy("person_id")
    .agg(
        count("*").alias("medication_count")
    )
)

display(medication_features)

In [0]:
visit_features = (
    visit_df
    .groupBy("person_id")
    .agg(
        count("*").alias("previous_admissions")
    )
)

display(visit_features)

In [0]:
visit_type_features = (
    visit_df
    .groupBy("person_id")
    .agg(
        first("visit_type").alias("visit_type")
    )
)

display(visit_type_features)

In [0]:
diagnosis_name_features = (
    condition_df
    .groupBy("person_id")
    .agg(
        first("condition_name").alias("primary_diagnosis")
    )
)

display(diagnosis_name_features)

In [0]:
ml_df = (
    encounter_df

    .join(person_features, "person_id", "left")

    .join(diagnosis_features, "person_id", "left")

    .join(medication_features, "person_id", "left")

    .join(visit_features, "person_id", "left")

    .join(visit_type_features, "person_id", "left")

    .join(diagnosis_name_features, "person_id", "left")

    .join(cohort_df, "person_id", "left")

    .join(readmission_df, "person_id", "left")
)

In [0]:
ml_df = ml_df.fillna({

    "age":0,

    "gender_concept_id":0,

    "diagnosis_count":0,

    "medication_count":0,

    "previous_admissions":0,

    "visit_type":"Unknown",

    "primary_diagnosis":"Unknown",

    "cohort_name":"Unknown",

    "readmission_30day_flag":0
})

In [0]:
display(ml_df)

print(ml_df.count())

ml_df.printSchema()

In [0]:
(
    ml_df.write
    .mode("overwrite")
    .saveAsTable(
        "db_healthcare_kl.gold.ml_readmission_features"
    )
)

In [0]:
ml_data = spark.table("db_healthcare_kl.gold.ml_readmission_features")

In [0]:
df = ml_data.select(
    "total_visits",
    "max_duration_days",
    "avg_duration_days",
    "age",
    "gender_concept_id",
    "diagnosis_count",
    "medication_count",
    "previous_admissions",
    "visit_type",
    "primary_diagnosis",
    "Cohort_name",
    "readmission_30day_flag"
)

In [0]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
df = df.na.drop(
    subset=["max_duration_days", "avg_duration_days"]
)

In [0]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()